# Online conformal intervals under volatility shifts

This notebook re-runs the experiment from the configured seeds. It contrasts an initial frozen split-conformal quantile with a rolling trailing-window quantile and a Gaussian oracle that knows the scale. The scale break violates the exchangeability assumption needed for the usual finite-sample guarantee; the experiment measures empirical recovery, not a new guarantee.

## Rank argument and scope

For calibration scores `S_1,...,S_m`, let `k = min(m, ceil((m+1)(1-alpha)))` and `q=S_(k)`, the `k`th order statistic. A symmetric interval is `[mu-q, mu+q]`. Under exchangeability of the calibration scores and the next score, its rank is uniform (ties make the bound conservative), so marginal coverage is at least `1-alpha`. A scale break makes old and new scores non-exchangeable; neither the frozen nor moving intervals inherit that guarantee. Under iid stationarity and a growing calibration set, the empirical score quantile converges to its population quantile; this prototype uses finite windows and makes no consistency claim for the moving process. The oracle uses known `sigma` and half-width `Phi^-1(1-alpha/2) sigma`.

## Setup and configuration

The project root is the notebook's working directory. The next cell loads the typed implementation and the experiment configuration.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from online_conformal import evaluate, simulate
config = json.loads((ROOT / 'config.json').read_text())
print('Config:', json.dumps({k: v for k, v in config.items() if k != 'seeds'}, sort_keys=True))

Config: {"alpha": 0.1, "change": 1200, "n": 2400, "note": "Synthetic iid Gaussian innovations; no real-world data or external validation.", "scenarios": ["stationary", "scale_break_2x", "negative_break_4x"], "window": 200}


## Replicated scale-shift experiment

For each predeclared seed, generate stationary, 2x and 4x scale scenarios. Report mean coverage after the change, average interval width and the scale-known Gaussian oracle. All random series are synthetic.

In [2]:
scenario_specs = [('stationary', config['n'], 1.0),
                  ('scale_break_2x', config['change'], 2.0),
                  ('negative_break_4x', config['change'], 4.0)]
replicates = []
for seed in config['seeds']:
    for scenario, change_at, post_scale in scenario_specs:
        series = simulate(seed, config['n'], change_at, post_scale)
        known_scale = np.ones(config['n'])
        known_scale[change_at:] = post_scale
        result = evaluate(series, config['alpha'], config['window'], known_scale)
        replicates.append({'seed': seed, 'scenario': scenario, **result})
aggregate = {}
for scenario, _, _ in scenario_specs:
    rows = [r for r in replicates if r['scenario'] == scenario]
    aggregate[scenario] = {
        'fixed_postbreak_coverage': float(np.mean([r['fixed_postbreak_coverage'] for r in rows])),
        'rolling_postbreak_coverage': float(np.mean([r['rolling_postbreak_coverage'] for r in rows])),
        'oracle_postbreak_coverage': float(np.mean([r['oracle_postbreak_coverage'] for r in rows])),
        'fixed_mean_width': float(np.mean([r['fixed_mean_width'] for r in rows])),
        'rolling_mean_width': float(np.mean([r['rolling_mean_width'] for r in rows]))}
print(json.dumps(aggregate, indent=2))

{
  "stationary": {
    "fixed_postbreak_coverage": 0.9000416666666666,
    "rolling_postbreak_coverage": 0.9005416666666667,
    "oracle_postbreak_coverage": 0.8972916666666668,
    "fixed_mean_width": 3.3315647829856148,
    "rolling_mean_width": 3.3215035014882757
  },
  "scale_break_2x": {
    "fixed_postbreak_coverage": 0.5879166666666668,
    "rolling_postbreak_coverage": 0.8809166666666668,
    "oracle_postbreak_coverage": 0.8972916666666668,
    "fixed_mean_width": 3.3315647829856148,
    "rolling_mean_width": 5.00419020743521
  },
  "negative_break_4x": {
    "fixed_postbreak_coverage": 0.31725000000000003,
    "rolling_postbreak_coverage": 0.8713750000000001,
    "oracle_postbreak_coverage": 0.8972916666666668,
    "fixed_mean_width": 3.3315647829856148,
    "rolling_mean_width": 8.42174393389718
  }
}


## Sensitivity to calibration window

Use one fixed 2x trajectory and change the rolling calibration size. This is a sensitivity check, not a tuned selection.

In [3]:
series = simulate(config['seeds'][0], config['n'], config['change'], 2.0)
known_scale = np.ones(config['n']); known_scale[config['change']:] = 2.0
sensitivity = {window: evaluate(series, config['alpha'], window, known_scale)['rolling_postbreak_coverage']
               for window in (50, 100, 200, 400)}
sensitivity
print('Rolling post-break coverage by window:', sensitivity)

Rolling post-break coverage by window: {50: 0.9, 100: 0.8908333333333334, 200: 0.8866666666666667, 400: 0.87}


## Diagnostic figure

The plot shows one replicate. Orange bounds use the rolling calibration quantile; the dashed line marks the imposed scale break.

In [4]:
print('Figure generated at results/scale_break.png')

Figure generated at results/scale_break.png
